In [34]:
import pandas as pd
import streamlit as st
import numpy as np

CCTV_FILE  = r"C:\Users\Win11Pro\Documents\과제\cctv\DATA\서울시 자치구 CCTV 설치현황.xlsx"
CRIME_CSV  = r"C:\Users\Win11Pro\Documents\과제\cctv\DATA\5대+범죄+발생현황_20260512140024.csv"
POP_FILE   = r"C:\Users\Win11Pro\Documents\과제\cctv\DATA\등록인구_20260512150716.xlsx"
LOC_FILE   = r"C:\Users\Win11Pro\Documents\과제\cctv\DATA\5대+범죄+발생장소별+현황.xlsx"


# ════════════════════════════════════════════════════════════════════════════════
# 데이터 로딩 (캐싱)
# ════════════════════════════════════════════════════════════════════════════════
@st.cache_data
def load_cctv():
    raw = pd.read_excel(CCTV_FILE, header=2)
    raw = raw.drop(columns=["Unnamed: 0"], errors="ignore")
    raw = raw.iloc[:-2]
    raw = raw[raw["구분"] != "계"].copy()
    year_cols = [c for c in raw.columns if str(c).endswith("년")]
    raw[year_cols] = raw[year_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
    return raw.set_index("순번")

@st.cache_data
def load_crime_csv():
    return pd.read_csv(CRIME_CSV, encoding="utf-8")


@st.cache_data
def load_population():
    return pd.read_excel(POP_FILE)


@st.cache_data
def load_crime_loc():
    return pd.read_excel(LOC_FILE)


@st.cache_data
def build_df_final():
    """2024년 기준 25개 자치구 통합 데이터프레임."""
    cctv      = load_cctv()
    crime_raw = load_crime_csv()
    pop_raw   = load_population()

    # ── 인구 (2024.1 = 총인구 / 2024 = 세대수로 오류 방지) ───────────────────
    pop_24 = pop_raw[["동별(2)", "2024.1"]].copy()
    pop_24.columns = ["자치구", "인구수"]
    pop_24["자치구"] = pop_24["자치구"].astype(str).str.strip()
    pop_24 = pop_24[~pop_24["자치구"].isin(["소계", "합계", "동별(2)"])]
    pop_24["인구수"] = pd.to_numeric(
        pop_24["인구수"].astype(str).str.replace(",", ""), errors="coerce"
    )
    pop_24 = pop_24.dropna(subset=["인구수"])
    pop_24 = pop_24[pop_24["인구수"] > 50_000]          # 동 단위 행 제거
    pop_24["인구수"] = pop_24["인구수"].astype(int)

    # ── CCTV (2024년) ─────────────────────────────────────────────────────────
    cctv_24 = cctv[["구분", "2024년"]].rename(
        columns={"구분": "자치구", "2024년": "CCTV수"}
    ).copy()
    cctv_24["CCTV수"] = pd.to_numeric(cctv_24["CCTV수"], errors="coerce")

    # ── 범죄 (2024년 소계 발생건수, 구별) ─────────────────────────────────────
    crime_filt = crime_raw[[c for c in crime_raw.columns if "." not in str(c)]].copy()
    crime_filt = crime_filt.drop(0).rename(columns={"자치구별(2)": "자치구"})
    if "자치구별(1)" in crime_filt.columns:
        crime_filt = crime_filt.drop(columns=["자치구별(1)"])
    crime_24 = crime_filt[["자치구", "2024"]].rename(columns={"2024": "발생건수"}).copy()
    crime_24["발생건수"] = pd.to_numeric(crime_24["발생건수"], errors="coerce")

    # ── 병합 ─────────────────────────────────────────────────────────────────
    df = pd.merge(pop_24, cctv_24, on="자치구")
    df = pd.merge(df, crime_24, on="자치구")
    df = df.dropna()

    # ── 지표 계산 ─────────────────────────────────────────────────────────────
    df["범죄율"]        = (df["발생건수"] / df["인구수"]) * 100_000   # 인구 10만명당
    df["CCTV_밀도"]     = (df["CCTV수"]   / df["인구수"]) * 10_000    # 인구 1만명당
    df["CCTV당_범죄수"] = df["발생건수"]  / df["CCTV수"]
    df.index = range(1, len(df) + 1)
    return df


@st.cache_data
def build_crime_trend():
    """2015–2024 서울 전체 연도별 CCTV·인구·5대 범죄 트렌드."""
    cctv      = load_cctv()
    crime_raw = load_crime_csv()
    pop_raw   = load_population()

    CRIME_TYPES   = ["살인", "강도", "강간·강제추행", "절도", "폭력"]
    CRIME_OFFSETS = [2, 4, 6, 8, 10]   # 발생 컬럼 오프셋

    rows = []
    for year in range(2015, 2025):
        yr = str(year)

        # 인구 (서울 전체 = 동별(2)=='소계', 총인구={yr}.1)
        try:
            pop_total = pd.to_numeric(
                str(pop_raw.loc[pop_raw["동별(2)"] == "소계", f"{yr}.1"].values[0])
                .replace(",", ""), errors="coerce"
            )
        except Exception:
            continue

        # CCTV 총합
        cctv_col = f"{yr}년"
        cctv_sum = cctv[cctv_col].sum() if cctv_col in cctv.columns else np.nan

        # 합계·소계 행
        row_sum = crime_raw[
            (crime_raw["자치구별(1)"] == "합계") &
            (crime_raw["자치구별(2)"] == "소계")
        ]
        if row_sum.empty:
            row_sum = crime_raw[crime_raw["자치구별(1)"] == "합계"].iloc[[0]]

        total_crime = pd.to_numeric(
            str(row_sum[yr].values[0]).replace(",", ""), errors="coerce"
        )
        crime_rate_total = (total_crime / pop_total) * 100_000

        type_rates = []
        for offset in CRIME_OFFSETS:
            col = f"{yr}.{offset}"
            val = pd.to_numeric(
                str(row_sum[col].values[0]).replace(",", ""), errors="coerce"
            )
            type_rates.append((val / pop_total) * 100_000)

        rows.append([yr, cctv_sum, pop_total, total_crime, crime_rate_total] + type_rates)

    df = pd.DataFrame(
        rows,
        columns=["연도", "CCTV수량", "인구수", "총범죄수", "총범죄율"] + CRIME_TYPES
    )
    df["연도"] = df["연도"].astype(str)
    return df


# ── 데이터 불러오기 ───────────────────────────────────────────────────────────
df_final  = build_df_final()
df_trend  = build_crime_trend()
cctv_raw  = load_cctv()

cctv_raw

2026-05-13 12:53:34.160 No runtime found, using MemoryCacheStorageManager
2026-05-13 12:53:34.162 No runtime found, using MemoryCacheStorageManager
2026-05-13 12:53:34.163 No runtime found, using MemoryCacheStorageManager
2026-05-13 12:53:34.163 No runtime found, using MemoryCacheStorageManager
2026-05-13 12:53:34.164 No runtime found, using MemoryCacheStorageManager
2026-05-13 12:53:34.167 No runtime found, using MemoryCacheStorageManager


,구분,2015년,2016년,2017년,2018년,2019년,2020년,2021년,2022년,2023년,2024년,2025년
순번,,,,,,,,,,,,
1,종로구,935.0,1066.0,1225.0,1322.0,1327.0,1510.0,1573.0,1812.0,1872.0,2154.0,2885.0
2,중구,363.0,565.0,838.0,1174.0,1242.0,1482.0,1911.0,2026.0,2157.0,2567.0,2861.0
3,용산구,1398.0,1689.0,1831.0,1888.0,1915.0,2058.0,2321.0,2531.0,2897.0,3202.0,3357.0
4,성동구,1089.0,1328.0,2103.0,2390.0,2833.0,3162.0,3519.0,3627.0,3871.0,4084.0,4226.0
5,광진구,638.0,657.0,1112.0,1586.0,2308.0,2481.0,3111.0,3370.0,3421.0,4370.0,4590.0
6,동대문구,1202.0,1425.0,1535.0,1775.0,2061.0,2166.0,2471.0,2592.0,3077.0,3602.0,4001.0
7,중랑구,751.0,898.0,1047.0,1203.0,2250.0,3165.0,3592.0,3856.0,4163.0,5009.0,5161.0
8,성북구,1035.0,1534.0,1940.0,2542.0,3238.0,3440.0,3815.0,4014.0,4216.0,4479.0,4994.0
9,강북구,608.0,840.0,841.0,1159.0,1656.0,2337.0,2960.0,3184.0,3191.0,3430.0,4434.0


In [ ]:
cctv_24 = cctv_raw[["구분", "2024년"]].rename(
    columns={"구분": "자치구", "2024년": "CCTV수"}
).copy()

cctv_24["CCTV수"] = pd.to_numeric(cctv_24["CCTV수"], errors="coerce")

cctv_24

,자치구,CCTV수
순번,,
1,종로구,2154.0
2,중구,2567.0
3,용산구,3202.0
4,성동구,4084.0
5,광진구,4370.0
6,동대문구,3602.0
7,중랑구,5009.0
8,성북구,4479.0
9,강북구,3430.0


In [5]:
cctv_origin = pd.read_excel(r"C:\Users\Win11Pro\Documents\과제\cctv\DATA\서울시 자치구 (범죄예방 수사용) CCTV 설치현황(25.12.31 기준).xlsx", skiprows=2)
cctv_seoul = cctv_origin[cctv_origin['구분'] != '계'].iloc[:25].copy()
cctv_seoul['구분'] = cctv_seoul['구분'].str.strip()
cctv_2024 = cctv_seoul[['구분', '2024년']].copy()
cctv_2024['2024년'] = pd.to_numeric(cctv_2024['2024년'].astype(str).str.replace(',', ''), errors='coerce')
cctv_2024.columns = ['지역구', 'CCTV수']
cctv_2024

,지역구,CCTV수
1,종로구,2154.0
2,중구,2567.0
3,용산구,3202.0
4,성동구,4084.0
5,광진구,4370.0
6,동대문구,3602.0
7,중랑구,5009.0
8,성북구,4479.0
9,강북구,3430.0
10,도봉구,2623.0


In [32]:
@st.cache_data
def load_data():
    df_cctv_raw = pd.read_excel("./DATA/서울시 자치구 (범죄예방 수사용) CCTV 설치현황(25.12.31 기준).xlsx", header=2)
    cctv_total = df_cctv_raw[df_cctv_raw['구분'] == '계']
    years_cctv = [f"{y}년" for y in range(2015, 2025)]
    cctv_vals = cctv_total[years_cctv].iloc[0].values

    crime_path = "crime(2015-2024).xlsx"
    headers = pd.read_excel(crime_path, header=None, nrows=4)
    data = pd.read_excel(crime_path, skiprows=4, header=None)
    df_util = pd.read_excel("util_clean.xlsx")
    
    return cctv_vals, headers, data, df_util

try:
    cctv_vals, crime_headers, crime_data, df_util = load_data()
except Exception as e:
    st.error(f"❌ 데이터 로드 실패: {e}")
    st.stop()

df_cctv_raw = pd.read_excel("./DATA/서울시 자치구 CCTV 설치현황.xlsx", header=2)
# df_cctv_raw = pd.read_excel("./DATA/서울시 자치구 (범죄예방 수사용) CCTV 설치현황(25.12.31 기준).xlsx", header=2)
cctv_total = df_cctv_raw[df_cctv_raw['구분'] == '계']
years_cctv = [f"{y}년" for y in range(2015, 2025)]
cctv_vals = cctv_total[years_cctv].iloc[0].values

df_cctv_raw

2026-05-13 12:51:44.475 No runtime found, using MemoryCacheStorageManager
2026-05-13 12:51:44.477 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-13 12:51:44.490 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-13 12:51:44.490 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-13 12:51:44.491 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-13 12:51:44.491 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


,Unnamed: 0,순번,구분,2015년,2016년,2017년,2018년,2019년,2020년,2021년,2022년,2023년,2024년,2025년
0,NaN,NaN,계,26321.0,33013.0,40512.0,49222.0,58139.0,67281.0,74408.0,80005.0,86810.0,99758.0,108947.0
1,NaN,1,종로구,935.0,1066.0,1225.0,1322.0,1327.0,1510.0,1573.0,1812.0,1872.0,2154.0,2885.0
2,NaN,2,중구,363.0,565.0,838.0,1174.0,1242.0,1482.0,1911.0,2026.0,2157.0,2567.0,2861.0
3,NaN,3,용산구,1398.0,1689.0,1831.0,1888.0,1915.0,2058.0,2321.0,2531.0,2897.0,3202.0,3357.0
4,NaN,4,성동구,1089.0,1328.0,2103.0,2390.0,2833.0,3162.0,3519.0,3627.0,3871.0,4084.0,4226.0
5,NaN,5,광진구,638.0,657.0,1112.0,1586.0,2308.0,2481.0,3111.0,3370.0,3421.0,4370.0,4590.0
6,NaN,6,동대문구,1202.0,1425.0,1535.0,1775.0,2061.0,2166.0,2471.0,2592.0,3077.0,3602.0,4001.0
7,NaN,7,중랑구,751.0,898.0,1047.0,1203.0,2250.0,3165.0,3592.0,3856.0,4163.0,5009.0,5161.0
8,NaN,8,성북구,1035.0,1534.0,1940.0,2542.0,3238.0,3440.0,3815.0,4014.0,4216.0,4479.0,4994.0
9,NaN,9,강북구,608.0,840.0,841.0,1159.0,1656.0,2337.0,2960.0,3184.0,3191.0,3430.0,4434.0
